# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method. 

In [1]:
%store -r data_set
%store -r labels
%store -r test_data_set
%store -r test_labels
%store -r unique_labels

## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [8]:
import numpy as np
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

In [10]:
def build_classifiers():
    candidate_classifiers = [
        ('Linear regression', LinearRegression()),
        ('Nearest Neighbors', KNeighborsClassifier()),
        ('Linear SVM', SVC(kernel='linear')),
        ('Decision Tree', DecisionTreeClassifier(random_state=15)),
        ('Naive Bayes', GaussianNB()),
        ('QDA', QuadraticDiscriminantAnalysis()),
    ]

    evaluated = []
    min_label = np.min(labels)
    max_label = np.max(labels)

    for name, clf in candidate_classifiers:
        clf.fit(data_set, labels)
        predicted = clf.predict(data_set)

        if name == 'Linear regression':
            predicted = np.rint(predicted)
            predicted = np.clip(predicted, min_label, max_label)

        score = accuracy_score(labels, predicted)
        evaluated.append((name, score, clf))

    evaluated.sort(key=lambda item: item[1], reverse=True)

    print('All classifiers (sorted by accuracy):')
    for name, score, _ in evaluated:
        print(f'{name}: {score:.6f}')

    top3_names = [name for name, _, _ in evaluated[:3]]
    print(f'TOP 3 classifiers: {top3_names}')

    best_three = [item[2] for item in evaluated[:3]]
    return tuple(best_three)


In [11]:
def build_stacked_classifier(classifiers):
    output = []
    for classifier in classifiers:
        output.append(classifier.predict(data_set))
    output = np.array(output).reshape((130,3))
    
    # stacked classifier part:
    stacked_classifier = DecisionTreeClassifier(random_state=15) # set here
    stacked_classifier.fit(output.reshape((130,3)), labels.reshape((130,)))
    test_set = []
    for classifier in classifiers:
        test_set.append(classifier.predict(test_data_set))
    test_set = np.array(test_set).reshape((len(test_set[0]),3))
    predicted = stacked_classifier.predict(test_set)
    return predicted

In [12]:
classifiers = build_classifiers()
predicted = build_stacked_classifier(classifiers)
accuracy = accuracy_score(test_labels, predicted)
print(accuracy)

All classifiers (sorted by accuracy):
Decision Tree: 1.000000
Linear SVM: 0.984615
QDA: 0.984615
Linear regression: 0.976923
Nearest Neighbors: 0.969231
Naive Bayes: 0.969231
TOP 3 classifiers: ['Decision Tree', 'Linear SVM', 'QDA']
0.85


## Exercise 2: 

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [14]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# prepare data set

def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels = np.random.choice(label_number, sample_number)
    return data_set, labels

labels = 2
dimension = 2
test_set_size = 1000
train_set_size = 5000
train_set, train_labels = generate_data(train_set_size, dimension, labels)
test_set, test_labels = generate_data(test_set_size, dimension, labels)

# init weights
number_of_iterations = 10
weights = np.ones((test_set_size,)) / test_set_size


def train_model(classifier, weights):
    return classifier.fit(X=test_set, y=test_labels, sample_weight=weights)

def calculate_error(model):
    predicted = model.predict(test_set)
    I=calculate_accuracy_vector(predicted, test_labels)
    Z=np.sum(I)
    return (1+Z)/1.0

Fill the two functions below:

def set_new_weights(model):
    # fill the code here (two lines)
    I = np.not_equal(model.predict(test_set), test_labels).astype(int)
    return (1 + I) / np.sum(1 + I)


In [15]:
def set_new_weights(model):
    # fill the code here (two lines)
    I = np.not_equal(model.predict(test_set), test_labels).astype(int)
    return (1 + I) / np.sum(1 + I)


Train the classifier with the code below:

In [16]:
classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)
alphas = []
classifiers = []
for iteration in range(number_of_iterations):
    model = train_model(classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print(weights)


validate_x, validate_label = generate_data(1, dimension, labels)

[0.00131406 0.00131406 0.00131406 0.00131406 0.00131406 0.00065703
 0.00131406 0.00131406 0.00131406 0.00065703 0.00131406 0.00065703
 0.00131406 0.00131406 0.00131406 0.00065703 0.00131406 0.00131406
 0.00131406 0.00131406 0.00131406 0.00065703 0.00065703 0.00131406
 0.00131406 0.00131406 0.00065703 0.00065703 0.00131406 0.00131406
 0.00131406 0.00131406 0.00131406 0.00131406 0.00065703 0.00065703
 0.00065703 0.00131406 0.00131406 0.00065703 0.00065703 0.00065703
 0.00131406 0.00065703 0.00131406 0.00131406 0.00065703 0.00065703
 0.00131406 0.00131406 0.00065703 0.00065703 0.00065703 0.00131406
 0.00065703 0.00131406 0.00065703 0.00131406 0.00065703 0.00131406
 0.00131406 0.00065703 0.00065703 0.00065703 0.00065703 0.00131406
 0.00065703 0.00131406 0.00131406 0.00131406 0.00131406 0.00131406
 0.00131406 0.00065703 0.00065703 0.00065703 0.00131406 0.00065703
 0.00131406 0.00065703 0.00131406 0.00065703 0.00065703 0.00065703
 0.00065703 0.00131406 0.00065703 0.00065703 0.00131406 0.0006

Set the validation data set:

In [17]:
validate_x, validate_label = generate_data(1, dimension, labels)

def get_prediction(x):
    # fill the code here (5-6 lines)
    votes = np.array([model.predict(x) for model in classifiers])
    predicted = []
    for i in range(votes.shape[1]):
        class_votes = np.bincount(votes[:, i].astype(int), minlength=labels)
        predicted.append(np.argmax(class_votes))
    return np.array(predicted)


In [18]:
def get_prediction(x):
    # fill the code here (5-6 lines)
    votes = np.array([model.predict(x) for model in classifiers])
    predicted = []
    for i in range(votes.shape[1]):
        class_votes = np.bincount(votes[:, i].astype(int), minlength=labels)
        predicted.append(np.argmax(class_votes))
    return np.array(predicted)


Test it:

In [19]:
prediction = get_prediction(validate_x)[0]

print(prediction)

1
